In [ ]:
import pandas as pd
df = pd.read_csv('listings.csv.gz', compression='gzip')

FileNotFoundError: [Errno 2] No such file or directory: 'listings.csv.gz'

In [3]:
import pandas as pd
df = pd.read_csv('listings.csv.gz', compression='gzip')

/tmp/ipykernel_2066/2598802007.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('listings.csv.gz', compression='gzip')


In [4]:
df.shape

(25715, 90)

In [5]:
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1293376226279904342,https://www.airbnb.com/rooms/1293376226279904342,20260916052113,2026-09-18,NaN,555 swanston,good location,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,509664637,...,5.0,5.0,4.0,NaN,NaN,2,0,2,0,0.05
1,1267716269242477675,https://www.airbnb.com/rooms/1267716269242477675,20260916052113,2026-09-18,NaN,On-suite - total privacy,"Relaxing and quite place, easy access to publi...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,473356895,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
2,1167644019888788983,https://www.airbnb.com/rooms/1167644019888788983,20260916052113,2026-09-17,NaN,Entertaining home,Get comfortable and enjoy plenty of extra room...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,22836099,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
3,1686698561880724365,https://www.airbnb.com/rooms/1686698561880724365,20260916052113,2026-09-18,NaN,Welcome home! Nestled in the heart of Melton.,Relax with the whole family at this peaceful p...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,1686691727730752808,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
4,1167133073069986715,https://www.airbnb.com/rooms/1167133073069986715,20260916052113,2026-09-18,NaN,Best fun place with best rates,"Kick back and relax in this calm, stylish space.",NaN,https://a0.muscache.com/pictures/miso/Hosting-...,377597884,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25715 entries, 0 to 25714
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            25715 non-null  int64  
 1   listing_url                                   25715 non-null  object 
 2   scrape_id                                     25715 non-null  int64  
 3   last_scraped                                  25715 non-null  object 
 4   source                                        6843 non-null   object 
 5   name                                          25715 non-null  object 
 6   description                                   25490 non-null  object 
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   25715 non-null  object 
 9   host_id                                       25715 non-null 

In [7]:
df.isnull().sum().sort_values(ascending=False).head(20)

,0
neighborhood_overview,25715
host_since,25715
host_response_time,25715
host_thumbnail_url,25715
host_acceptance_rate,25715
host_response_rate,25715
host_verifications,25715
neighbourhood,25715
host_total_listings_count,25715
host_neighbourhood,25715


## Initial findings
- 15 columns are completely empty (25,715 missing values each) and will be dropped.
- 6,711 listings (26%) have no price, likely inactive. These will be excluded from the pricing analysis.

In [8]:
empty_cols = df.columns[df.isnull().all()]
print(len(empty_cols))
print(empty_cols.tolist())

15
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated', 'license', 'instant_bookable']


- Price is stored as text ("$80.00"); needs converting to a number before analysis.

In [9]:
print(df['price'].head(10))

0     $80.00
1    $100.00
2        NaN
3     $56.00
4     $90.00
5    $453.00
6    $216.00
7    $185.00
8    $326.00
9    $633.00
Name: price, dtype: object


In [10]:
print(df['neighbourhood_cleansed'].value_counts().head(10))

neighbourhood_cleansed
Melbourne       8422
Port Phillip    2761
Yarra           1785
Stonnington     1308
Yarra Ranges    1081
Moreland         959
Wyndham          823
Maribyrnong      651
Whitehorse       651
Monash           627
Name: count, dtype: int64


- Neighbourhoods are council areas (LGAs), not suburbs. City of Melbourne holds ~33% of all listings.
- Yarra Ranges (5th) suggests a separate tourism/getaway market worth comparing to inner-city areas.

In [11]:
print(df['room_type'].value_counts())

room_type
Entire home/apt    18959
Private room        6475
Shared room          227
Hotel room            54
Name: count, dtype: int64


- 74% of listings are entire homes/apts. Analysis will focus on these, as the relevant product for a property investor. Shared and hotel rooms (~1%) excluded.